# 04. 기상청 ASOS 시간자료

입력: `data/processed/segments.csv`, `data/interim/accident_clean.csv`, `data/external/asos_stations.csv`
출력: `data/external/asos_hourly_all.csv`, `data/external/asos_stations_used.csv`

**준비**
1. data.go.kr에서 **15057210 기상청_지상(종관, ASOS) 시간자료 조회서비스** 활용신청 (같은 키)
2. ASOS 관측소 목록(지점번호·위도·경도)을 `data/external/asos_stations.csv`로 저장 (기상자료개방포털 지점정보)

사고가 없는 시간대(네거티브 샘플)에도 기상이 필요해서, 고속도로 전체 구간 근처 관측소의 **전체 기간** 시간자료를 받아요.
개발계정은 하루 10,000회 제한이 있어요. 중간에 멈추면 다음 날 다시 실행하면 받은 건 건너뛰고 이어서 받아요.

| 열 | 의미 |
|---|---|
| ss 일조 | 1시간 중 해가 비친 시간(hr) |
| icsr 일사 | 1시간 동안 받은 태양 에너지(MJ/m²) |
| dc10Tca 전운량 | 하늘을 덮은 구름 (0~10) |
| vs 시정 | 앞이 보이는 거리 (10m 단위) |
| ws 풍속, rn 강수, ta 기온, hm 습도, dsnw 적설, ts 지면온도 | 노면·주행 조건 |

In [ ]:
%pip install requests scikit-learn -q

In [ ]:
import os, time
from pathlib import Path
import numpy as np
import pandas as pd
import requests
from sklearn.neighbors import BallTree

ROOT = Path.cwd()
INTERIM, EXTERNAL, PROCESSED = (ROOT / "data" / d for d in ["interim", "external", "processed"])
CACHE = EXTERNAL / "asos_hourly"
CACHE.mkdir(parents=True, exist_ok=True)

def get_service_key():
    key = os.environ.get("DATA_GO_KR_KEY")
    if not key and (ROOT / ".env").exists():
        for line in (ROOT / ".env").read_text(encoding="utf-8").splitlines():
            if line.strip().startswith("DATA_GO_KR_KEY="):
                key = line.split("=", 1)[1].strip().strip('"').strip("'")
    assert key, ".env 에 키가 없습니다"
    return key
SERVICE_KEY = get_service_key()

## 1. 고속도로 구간마다 가장 가까운 관측소

In [ ]:
st = pd.read_csv(EXTERNAL / "asos_stations.csv", encoding_errors="replace")
col = lambda key: next(c for c in st.columns if key in c)   # 열 이름이 조금 달라도 찾도록
st = st.rename(columns={col("지점"): "stn_id", col("위도"): "stn_lat", col("경도"): "stn_lon"})
st = st.drop_duplicates("stn_id", keep="last")[["stn_id", "stn_lat", "stn_lon"]].reset_index(drop=True)

seg = pd.read_csv(PROCESSED / "segments.csv")
tree = BallTree(np.radians(st[["stn_lat", "stn_lon"]]), metric="haversine")
dist, idx = tree.query(np.radians(seg[["mid_lat", "mid_lon"]]), k=1)
used = st.iloc[np.unique(idx[:, 0])]
used.to_csv(EXTERNAL / "asos_stations_used.csv", index=False)
print(f"관측소 {len(st)}개 중 고속도로 근처 {len(used)}개 사용, 구간→관측소 거리 중앙값 {np.median(dist) * 6371:.1f} km")

## 2. 관측소 × 월 단위로 받기

In [ ]:
URL = "https://apis.data.go.kr/1360000/AsosHourlyInfoService/getWthrDataList"

def fetch_month(stn_id, month_start):
    out = CACHE / f"{stn_id}_{month_start:%Y%m}.csv"
    if out.exists():
        return
    month_end = min(month_start + pd.offsets.MonthEnd(0), pd.Timestamp.today().normalize() - pd.Timedelta(days=1))
    params = {"serviceKey": SERVICE_KEY, "pageNo": 1, "numOfRows": 999, "dataType": "JSON",
              "dataCd": "ASOS", "dateCd": "HR", "stnIds": stn_id,
              "startDt": f"{month_start:%Y%m%d}", "startHh": "00", "endDt": f"{month_end:%Y%m%d}", "endHh": "23"}
    for attempt in range(3):
        try:
            r = requests.get(URL, params=params, timeout=30)
            body = r.json()["response"]
            break
        except Exception as e:
            if attempt == 2:
                raise RuntimeError(f"{stn_id} {month_start:%Y-%m} 실패: {r.text[:300]}") from e
            time.sleep(2 ** attempt)
    if body["header"]["resultCode"] != "00":
        raise RuntimeError(f"{stn_id} {month_start:%Y-%m}: {body['header']['resultMsg']}")
    pd.DataFrame(body["body"]["items"]["item"]).to_csv(out, index=False, encoding="utf-8-sig")
    time.sleep(0.1)

acc = pd.read_csv(INTERIM / "accident_clean.csv", parse_dates=["사고일시"])
months = pd.date_range(acc.사고일시.min().to_period("M").to_timestamp(), acc.사고일시.max(), freq="MS")
todo = [(s, m) for s in used.stn_id for m in months if not (CACHE / f"{s}_{m:%Y%m}.csv").exists()]
print(f"남은 호출 {len(todo)}회 (관측소 {len(used)} × {len(months)}개월)")
for n, (s, m) in enumerate(todo, 1):
    fetch_month(s, m)
    if n % 100 == 0:
        print(f"  {n}/{len(todo)}")
print("완료")

## 3. 하나로 합치기

In [ ]:
COLS = ["ta", "rn", "ws", "hm", "ss", "icsr", "dc10Tca", "vs", "dsnw", "ts"]
w = pd.concat([pd.read_csv(f, dtype=str) for f in CACHE.glob("*.csv")], ignore_index=True)
w["tm"] = pd.to_datetime(w["tm"])
w["stn_id"] = w["stnId"].astype(int)
for c in COLS:
    w[c] = pd.to_numeric(w.get(c), errors="coerce")
w[["rn", "ss", "icsr", "dsnw"]] = w[["rn", "ss", "icsr", "dsnw"]].fillna(0)   # 빈칸 = 없음(0)
w = w[["stn_id", "tm", *COLS]].drop_duplicates(["stn_id", "tm"])
w.to_csv(EXTERNAL / "asos_hourly_all.csv", index=False, encoding="utf-8-sig")
print(w.shape)
w.describe().T.round(2)